# GLM-5.3-Flash EXL3 4bpw on 2× DGX Spark (GB10) — 1M-token window, TensorFold

**Verdict.** On two Sparks, TensorFold serves GLM-5.3-Flash 4-bit EXL3 with a **1,048,576-token window** at **54 tok/s decode** (short prompt, thinking off) and **~1,500–1,860 tok/s prefill**: **3.6× the decode and 4–11× the prefill** of the one-Spark 2-bit llama.cpp run. A **245k-token prompt takes 2.6 min** instead of 28.8 min. Decode rows marked * use copy-heavy prompts and are inflated by copy drafts.

In [1]:
# --- Status cell -------------------------------------------------------
# LIVE = False replays the committed receipts under results/<experiment>/.
# LIVE = True runs the same harness against a running endpoint whose URL
# comes from the environment. Never commit a notebook executed with LIVE = True.
import os, json
EXPERIMENT = "2026-10-01-glm-5.3-flash-exl3-2node-tensorfold"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT)
RECEIPTS = os.path.join(RESULTS_DIR, "receipts")
LIVE = False
print(f"experiment : {EXPERIMENT}\nLIVE       : {LIVE}")
if LIVE and not os.environ.get("GLM_ENDPOINT"):
    raise RuntimeError("LIVE=True but GLM_ENDPOINT is not set")
def receipt(name):
    with open(os.path.join(RECEIPTS, name)) as f:
        return json.load(f)

experiment : 2026-10-01-glm-5.3-flash-exl3-2node-tensorfold
LIVE       : False


## 1. TL;DR

| Metric | TensorFold, 2× Spark, 4-bit | llama.cpp, 1× Spark, 2-bit | Label |
|---|---|---|---|
| Window | **1,048,576 tokens**, 4 requests share an FP8 pool | 262,144, 1 request | configured |
| Decode, short prompt | **54.4** tok/s (thinking off), 67.2 (on) | 15.1 / 15.7 | measured |
| Prefill | 1,494 @ 7k, 1,865 @ 28k, 1,728 @ 128k, 1,553 @ 245k | 378 / 346 / 212 / 141 | measured |
| TTFT at 245k | **2.6 min** | 28.8 min | measured (n=1) |
| Memory after load | ~111 GiB used per Spark, 8–10 GiB free | 106 GB used, ~4 GB free at 245k | measured |
| Quality | not scored here | not scored | untested |

| Pin | Value |
|---|---|
| Recipe | `MiaAI-Lab/GLM-5.3-Flash-EXL3-2x-DGX-Sparks-TensorFold` @ `ed026ef9` |
| Model | `Mia-AiLab/GLM-5.3-Flash-EXL3-TR3-4bpw` @ `9eaebb7c` |
| Drafter | `incoai/GLM-5.3-Flash-DFlash2` @ `bf582e4e` (CC BY-NC-ND 4.0) |
| Image | `tensorfold:v0.5.0-cefe8bf45d07` |

## 2. Visible results

In [2]:
b = receipt("bench.json")
rows = []
for k, v in {**b["short_and_mid"], **b["long"]}.items():
    rows.append((k, v.get("prompt_tokens"), v.get("ttft_s"), v.get("prefill_tok_s"), v.get("decode_tok_s")))
print(f"{'workload':28}{'prompt tok':>11}{'TTFT s':>9}{'prefill t/s':>13}{'decode t/s':>12}")
for k, p, t, pf, d in rows:
    f = lambda x, n=1: "-" if x is None else f"{x:.{n}f}"
    print(f"{k:28}{p:>11.0f}{f(t):>9}{f(pf, 0):>13}{f(d):>12}")
print("\nmethod:", b["method"])
print("note: decode after 28k / 128k / 245k uses copy-heavy prompts; copy drafts inflate those rows")

workload                     prompt tok   TTFT s  prefill t/s  decode t/s
decode_short_think_off               30      0.2            -        54.4
prefill_7k                         7433      5.0         1494           -
prefill_28k                       27783     14.9         1865           -
decode_at_28k_think_off           27717     15.0            -       114.3
decode_short_think_on                22      0.2            -        67.2
ctx_128k                         127724     73.9         1728        56.6
ctx_245k                         244136    157.2         1553        66.0

method: temperature 0; median of 3 (n=1 for 128k and 245k); unique random prefix per prompt; tokens from the final usage object; client on a separate machine over the network; thinking off unless the label says on; one request at a time
note: decode after 28k / 128k / 245k uses copy-heavy prompts; copy drafts inflate those rows


## 3. Reproduce

Hardware: two DGX Sparks with nothing else on their GPUs (~110 GiB free each), a QSFP cable between their ConnectX-7 ports with an IPv4 address on each end, and key-based ssh from the head to the worker. Files are in `results/2026-10-01-glm-5.3-flash-exl3-2node-tensorfold/harness/`.

1. `git clone https://github.com/MiaAI-Lab/GLM-5.3-Flash-EXL3-2x-DGX-Sparks-TensorFold && cd GLM-5.3-Flash-EXL3-2x-DGX-Sparks-TensorFold && git checkout ed026ef92d1650120dada1294a112acb6c8f2f48`
2. `cp scripts/local.sh.example scripts/local.sh` and set `WORKER=user@<worker CX7 address>`.
3. If the two Sparks use different Docker image stores, `git apply harness/image-id-check.patch`.
4. `scripts/prepare.sh` (image on both Sparks; ~176 GB checkpoint + DFlash2 downloaded on the head and copied to the worker), then `./start.sh` until `GLM-5.3-Flash-EXL3 is now LIVE! on port 8888`. The first start compiles CUDA kernels; later starts load in 2–6 min.
5. Measure: `python3 harness/bench.py http://HOST:8888/v1 GLM-5.3-Flash-EXL3 bench.json`, then `URL=http://HOST:8888/v1 MODEL=GLM-5.3-Flash-EXL3 python3 harness/long.py`.

In [3]:
env = receipt("env.json")
for k, v in env.items():
    print(f"{k}: {v}")

hardware: 2 x DGX Spark (GB10, SM121, 128 GB unified), one QSFP ConnectX-7 cable (one rail cabled), driver 580.159.03
runtime: TensorFold v0.5.0 + 52 patches, MiaAI-Lab/GLM-5.3-Flash-EXL3-2x-DGX-Sparks-TensorFold @ ed026ef92d1650120dada1294a112acb6c8f2f48
image: ghcr.io/miaai-lab/glm-5.3-flash-exl3-2x-dgx-sparks-tensorfold:v0.5.0-cefe8bf45d07 (NVIDIA pytorch 26.07-py3 base)
model: Mia-AiLab/GLM-5.3-Flash-EXL3-TR3-4bpw @ 9eaebb7c4e96d983dcd538e18624622ba5b820a8
drafter: incoai/GLM-5.3-Flash-DFlash2 @ bf582e4eacc1810f76656d1811693ff6c6737d2a (CC BY-NC-ND 4.0), DRAFT_POLICY fnc7:0.3 + copy drafts
settings: recipe defaults: PARALLEL=4, CONTEXT=1048576, KV=fp8, DENSE=q4, VISION=1
memory_after_load_GiB: {'rank0_used_of_119': 111, 'rank0_available': 8, 'rank1_used_of_121': 111, 'rank1_available': 10}
restart_count_after_tests: 0
local_change: harness/image-id-check.patch: prepare.sh compared docker image IDs, which differ when one node uses the containerd image store and the other overlay2; c

## Appendix: failures and limits

- `prepare.sh` refused the worker's image (`differs from the head's`) although every layer digest matched; the two Sparks used different Docker image stores. Patched to compare layer digests.
- The DFlash2 drafter is CC BY-NC-ND 4.0 (non-commercial). `DRAFTER=mtp` avoids it, at one request at a time.
- Concurrency (2–4 requests), vision, and quality were not measured in this run; the recipe's own figures are cited, not reproduced.
- 8–10 GiB host memory remains on each Spark after load; do not co-locate another job.